# 02 -- GARCH Baseline Analysis

**Project:** Algorithmic Trading Using Time Series Volatility Signals and MARL  
**Author:** Aditya Raj Singh  

This notebook presents the GARCH baseline results (Section 4.1 of the paper).

**Models evaluated:**
1. GARCH(1,1) with Student-t innovations
2. EGARCH(1,1) with Student-t innovations (asymmetric)
3. GJR-GARCH(1,1) with Student-t innovations (leverage)
4. HAR-RV -- Heterogeneous Autoregressive Realised Volatility (Corsi, 2009)

**Evaluation:**
- Out-of-sample rolling forecasts on 2024 test data (refit every 22 days)
- Primary metric: QLIKE (Patton, 2011)
- Diebold-Mariano test for statistical significance

All figures are pre-generated by `run_02_garch_analysis.py` and saved to
`results/figures/garch/`.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import TABLES_DIR, FIGURES_DIR

GARCH_FIG_DIR = FIGURES_DIR / "garch"
print(f"Tables: {TABLES_DIR}")
print(f"Figures: {GARCH_FIG_DIR}")

## 1. Model Comparison -- All 20 Tickers

Average out-of-sample metrics across 10 NSE and 10 NASDAQ stocks.

In [ ]:
results = pd.read_csv(TABLES_DIR / "garch_baseline_results.csv")
avg = results.groupby("Model")[["RMSE", "MAE", "QLIKE", "R2"]].mean().round(4)
print("Average metrics across all 20 tickers:")
print(avg.sort_values("QLIKE").to_string())
print(f"\nBest model by QLIKE: {avg['QLIKE'].idxmin()} ({avg['QLIKE'].min():.4f})")

In [ ]:
# Win counts
best_rows = results[results["Best"] == "*"]
print("Model win counts (lowest QLIKE per ticker):")
print(best_rows["Model"].value_counts().to_string())

## 2. Diebold-Mariano Test

Testing whether HAR-RV significantly outperforms GARCH(1,1).

In [ ]:
dm = pd.read_csv(TABLES_DIR / "diebold_mariano_tests.csv")
print("Diebold-Mariano Test: GARCH(1,1) vs HAR-RV")
print("H0: Equal predictive accuracy")
print("Positive DM stat => GARCH(1,1) has larger squared error\n")
print(dm.to_string(index=False))

## 3. Key Findings

### Finding 1: HAR-RV dominates all GARCH variants

HAR-RV wins on **all 20 tickers** by QLIKE, with an average QLIKE improvement
of ~10% over the best GARCH variant (EGARCH). This is consistent with the
volatility forecasting literature (Corsi, 2009; Andersen et al., 2003).

### Finding 2: Asymmetric GARCH models show marginal improvement

EGARCH slightly outperforms symmetric GARCH and GJR-GARCH, confirming
the leverage effect (negative shocks increase volatility more than positive
ones). However, the improvement is small (~0.8% QLIKE).

### Finding 3: All individual models show negative R-squared

The negative out-of-sample R-squared across all models indicates that
none of these linear/parametric approaches fully captures the non-linear
dynamics of realised volatility. This **motivates the CNN-GARCH hybrid**
in Stage 1 -- the CNN can learn non-linear patterns that GARCH misses.

### Finding 4: HAR-RV's advantage is statistically significant

Diebold-Mariano tests reject equal predictive accuracy at p < 0.05 for
all 4 representative stocks, confirming HAR-RV is the **strongest baseline**
the CNN-GARCH model must beat.

### Implication for the CNN-GARCH Hybrid

The CNN model should be designed to:
1. **Combine HAR-RV features with learned non-linear patterns** from raw returns
2. **Use GARCH conditional variance as an input feature** (it captures time-varying
   volatility dynamics even if its point forecasts are weaker than HAR-RV)
3. **Target outperforming HAR-RV QLIKE of 0.509** as the success criterion